# Colab smoke: Qucs-S AppImage + one real simulation

**Goal:** Prove Colab can run this repo's reward path (`qucs-s -n` → `qucsator_rf`) **without** layout export.

1. Runtime type: **GPU optional** for this smoke (CPU is enough).
2. Run all cells top-to-bottom.
3. Success = last cell prints finite `|S21|` dB at 5.5 GHz and does not raise.

Uses [Qucs-S 26.1.1 Linux AppImage](https://github.com/ra3xdh/qucs_s/releases/tag/26.1.1).

**Note:** This AppImage's Qt only ships the `xcb` platform plugin (not `offscreen`), so Colab needs `xvfb`.


## 1) Clone repo (or skip if already mounted from Drive)


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/isaacguo/qucs-llm-optimizer.git"
REPO_DIR = Path("/content/qucs-llm-optimizer")

if not (REPO_DIR / "src" / "qucs_sim.py").exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print("repo already present:", REPO_DIR)
    !git -C {REPO_DIR} pull --ff-only || true

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())


## 2) Install Xvfb (required: AppImage Qt has xcb only, not offscreen)


In [ ]:
!apt-get -qq update
!apt-get -qq install -y xvfb libxkbcommon-x11-0 libxcb-xinerama0 libxcb-cursor0
!which Xvfb && Xvfb -help 2>&1 | head -1


## 3) Download + extract Qucs-S AppImage (no FUSE)


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

QUCS_DIR = Path("/content/qucs-s-appimage")
QUCS_DIR.mkdir(parents=True, exist_ok=True)

!python scripts/setup_qucs_appimage.py --dir {QUCS_DIR}


## 4) Point resolvers at binaries + start virtual display


In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / "src"))

from scripts.qucs_appimage import apply_qucs_env, find_qucs_binaries

extract = Path("/content/qucs-s-appimage/squashfs-root")
binaries = find_qucs_binaries(extract)
env = apply_qucs_env(binaries, extract_root=extract)
for k in ("QUCS_S", "QUCSATOR_RF", "DISPLAY", "QT_QPA_PLATFORM", "QT_PLUGIN_PATH", "LD_LIBRARY_PATH"):
    print(f"{k}={os.environ.get(k)}")

from qucs_sim import resolve_qucs_s, resolve_qucsator

print("resolve_qucs_s:", resolve_qucs_s())
print("resolve_qucsator:", resolve_qucsator())


## 5) One real simulation (no layout) + cost at 5.5 GHz


In [ ]:
import math
from pathlib import Path

from cost import evaluate, s21_db
from intent import INITIAL_GUESS
from qucs_sim import simulate

workdir = Path("/content/qucs-smoke-run")
result = simulate(dict(INITIAL_GUESS), workdir=workdir, export_layout=False)
cost = evaluate(result, (4e9, 6e9), target_hz=5.5e9)

db = s21_db(cost.total_cost)
assert math.isfinite(db), f"non-finite dB: {db}"
assert (workdir / "circuit.net").exists()
assert (workdir / "circuit.dat").exists()
assert not (workdir / "layout.svg").exists(), "layout should be skipped"

print("SMOKE OK")
print(f"points={len(result.freq_hz)}  |S21|@5.5GHz={cost.total_cost:.6g} ({db:.3f} dB)")
print(f"workdir={workdir}")


## If this fails

| Symptom | Likely cause |
|---------|--------------|
| AppImage extract fails | Missing libs / glibc — paste full traceback |
| `qucs-s` / `qucsator_rf` not found | AppImage layout changed — list `squashfs-root` |
| `no Qt platform plugin "offscreen"` | Stale env; re-run cells 2+4 so `QT_QPA_PLATFORM=xcb` + Xvfb |
| `could not connect to display` | Xvfb not installed/started — re-run cell 2 and 4 |
| Simulation OK here, training later OOM | Separate Unsloth/GPU issue |
